In [14]:
#Om Korat
#BIT 3474
#Oct 14, 2026

#importing everything I need
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup as soup
import csv
import time

#Turning the department urls into variables
bit_og = requests.get('https://bit.vt.edu/faculty/directory.html')
fin_og = requests.get('https://finance.pamplin.vt.edu/faculty/directory.html')
marketing_og = requests.get('https://marketing.pamplin.vt.edu/people/faculty.html')
OG = [bit_og, fin_og, marketing_og]

#Printing the service status for the directory urls
for og in OG:
  if og is None:
    print("N/A")
  else:
    print(og.status_code)

#Parse the HTML content of a specific page
bit = soup(bit_og.content, 'html.parser')
fin = soup(fin_og.content, 'html.parser')
marketing = soup(marketing_og.content, 'html.parser')
department = [bit, fin, marketing]

#Extracting each faculty url from the department and storing the links in an array
links = []
for de in department:
  if de is not None:
    prof_link = de.find_all("a", class_="vt-list-item-title-link")
    for prof in prof_link:
      url = prof.get('href')
      if url:
        # Ensure we construct valid absolute URLs if needed
        if url.startswith('/'):
          url = 'https://pamplin.vt.edu' + url
        links.append(url)

#Using a loop to parse the html content in the links
Profs = []
for link in links:
  if link is None or link == "Not available":
    continue
  try:
    faculty = requests.get(link, timeout = 10)
    if faculty.status_code == 200:
      prof = soup(faculty.content, 'html.parser')
      Profs.append(prof)
  except Exception as e:
    print(f"Error fetching {link}: {e}")
  time.sleep(0.1)

#Making a list for all of the data
Alldepart = []
Allcol = []
Alllast = []
Allfirst = []
Alltitle = []
Alladdress = []
Allemail = []
Allphone = []

#Finding the department on the instructor page
for profs in Profs:
  crumb = profs.find_all("a", class_="vt-breadcrumbs-link")
  if crumb is None or len(crumb) < 2:
    Alldepart.append("Not Available")
  else:
    depart = crumb[1].get_text(strip=True)
    Alldepart.append(depart)

#Finding the college on the instructor page
for profs in Profs:
  bread = profs.find("a", class_="vt-breadcrumbs-link")
  if bread is None:
    Allcol.append("Not Available")
  else:
    col = bread.get("title", "Not Available")
    Allcol.append(col)

#Finding the first and last name on the instructor page
for profs in Profs:
  name = profs.find("h1", class_="vt-bio-name")
  if name is None:
    Alllast.append("Not Available")
    Allfirst.append("Not Available")
  else:
    part = name.text.strip().split()
    if len(part) >= 2:
      Alllast.append(part[-1]) # Usually last name is at the end
      Allfirst.append(part[0])
    elif len(part) == 1:
      Alllast.append(part[0])
      Allfirst.append("Not Available")
    else:
      Alllast.append("Not Available")
      Allfirst.append("Not Available")

#Finding the title on the instructor page
for profs in Profs:
  title = profs.find("div", class_="vt-person-title")
  if title is None:
    Alltitle.append("Not Available")
  else:
    Alltitle.append(title.text.strip())

#Finding the office building and number on the instructor page
for profs in Profs:
  address_raw = profs.find("address", class_="vt-bio-contact-address")
  if address_raw is None:
    Alladdress.append("Not Available")
  else:
    address = address_raw.get_text(", ", strip=True)
    Alladdress.append(address)

#Finding the email on the instructor page
for profs in Profs:
  email = profs.find("div", class_="vt-bio-email")
  if email is None:
    Allemail.append("Not Available")
  else:
    Allemail.append(email.text.strip())

#Finding the phone on the instructor page
for profs in Profs:
  phone = profs.find("div", class_="vt-bio-phone")
  if phone is None:
    Allphone.append("Not Available")
  else:
    Allphone.append(phone.text.strip())

#Naming all the variables to put into a csv
Allinfo = {
  "Department": Alldepart,
  "College": Allcol,
  "Last Name": Alllast,
  "First Name": Allfirst,
  "Office Location": Alladdress,
  "Title": Alltitle,
  #"Email": Allemail,
  #"Phone": Allphone
}
csv_filename = "omkorat_web_scraping.csv"

#Putting all the data into a csv
df = pd.DataFrame(Allinfo)
df.to_csv(csv_filename, index=False)

#Putting an an auto-incrementing uniqueID into the csv
read = pd.read_csv("omkorat_web_scraping.csv")
read.insert(0, "UniqueID", range(1, 1 + len(read)))
read.to_csv("omkorat_web_scraping.csv", index=False)

200
200
200
